# 4-Way Multi-Scale Super-Resolution Evaluation (10m → 2.5m → 1.25m)
### B.Tech Major Project: Analytical Benchmarking Across 4 Spatial Scales

This notebook executes the end-to-end multi-scale comparative evaluation comparing:
1. **Normal Captured Input (10m)**: Unprocessed Sentinel-2 L2A tile upscaled bilinearly 8×.
2. **Pretrained Baseline SEN2SR Lite (2.5m)**: Standard 4× baseline model from Taco Foundation.
3. **Fine-Tuned SEN2SR Lite (2.5m, Ours)**: Stage 1 generator fine-tuned with Sobel edge & spectral loss.
4. **Cascaded Ultra-Res Stage 2 (1.25m, Ours)**: End-to-end progressive 8× super-resolution reaching ~1m ground fidelity.

Evaluated across **Overall Multispectral**, **RGB**, **NDVI**, **NDWI**, and **NDBI** with spatial gradient metrics and difference heatmaps.

## Step 1 — Mount Google Drive & Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, re, glob, zipfile, shutil, time, warnings, logging
warnings.filterwarnings('ignore')
for l_name in ['rasterio', 'rasterio._env', 'rasterio.env']:
    l = logging.getLogger(l_name)
    l.setLevel(logging.ERROR)
    l.propagate = False

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

BASE_DIR = '/content/drive/MyDrive/Project_Folder'
DATASETS_DIR = os.path.join(BASE_DIR, 'Datasets')
OUTPUT_DIR = os.path.join(BASE_DIR, 'Results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

STAGE1_CKPT = os.path.join(OUTPUT_DIR, 'sen2sr_lite_compound_finetuned.pth')
STAGE2_CKPT = os.path.join(OUTPUT_DIR, 'sen2sr_cascaded_1meter_refiner.pth')
print(f'Stage 1 Checkpoint: {STAGE1_CKPT} (Exists: {os.path.exists(STAGE1_CKPT)})')
print(f'Stage 2 Checkpoint: {STAGE2_CKPT} (Exists: {os.path.exists(STAGE2_CKPT)})')

## Step 2 — Install Required Libraries

In [ ]:
%pip install -q sen2sr mlstac rasterio safetensors pandas matplotlib

## Step 3 — Extract Copernicus Input ZIP

In [ ]:
# Specify input ZIP file (change to Browser_images_2.zip, etc. as needed)
INPUT_ZIP_NAME = 'Browser_images_1.zip'

ZIP_PATH = None
target_path = os.path.join(DATASETS_DIR, INPUT_ZIP_NAME)
if os.path.exists(target_path):
    ZIP_PATH = target_path
else:
    found = sorted(glob.glob(os.path.join(DATASETS_DIR, 'Browser_images*.zip')) + glob.glob('/content/Browser_images*.zip'))
    if found:
        ZIP_PATH = found[-1]

if not ZIP_PATH:
    raise FileNotFoundError(f'Could not find any Browser_images ZIP in {DATASETS_DIR}')

print(f'Using input ZIP archive: {ZIP_PATH}')
EXTRACT_DIR = '/content/copernicus_data'
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
    archive.extractall(EXTRACT_DIR)

raster_files = []
for root, _, files in os.walk(EXTRACT_DIR):
    for f in files:
        if f.lower().endswith(('.tif', '.tiff', '.jp2')):
            raster_files.append(os.path.join(root, f))
print(f'Extracted {len(raster_files)} raster files.')

## Step 4 — Align and Stack All 10 Sentinel-2 Bands

In [ ]:
import rasterio
from rasterio.warp import reproject, Resampling

BAND_NAMES = ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']
found_bands = {}
for path in raster_files:
    upper = os.path.basename(path).upper()
    matches = [b for b in BAND_NAMES if re.search(rf'(^|[_.\-]){b}([_.\-]|$)', upper)]
    if matches:
        found_bands[matches[0]] = path

missing = [b for b in BAND_NAMES if b not in found_bands]
if missing:
    raise ValueError(f'Missing required bands: {missing}')

# Reference grid is B02 (10m resolution)
with rasterio.open(found_bands['B02']) as ref:
    ref_transform, ref_crs = ref.transform, ref.crs
    height, width = ref.height, ref.width

stack = []
for b in BAND_NAMES:
    with rasterio.open(found_bands[b]) as src:
        dest = np.zeros((height, width), dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dest,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=Resampling.bilinear
        )
        stack.append(dest)

raw_stack = np.stack(stack).astype(np.float32)
finite_vals = raw_stack[np.isfinite(raw_stack)]
max_val = np.percentile(finite_vals, 99)
scale_factor = 10000.0 if max_val > 255.0 else (255.0 if max_val > 2.0 else 1.0)
X_np = np.nan_to_num(raw_stack / scale_factor, nan=0.0, posinf=1.0, neginf=0.0)
X_10band = torch.from_numpy(X_np).float().to(device)
print(f'10-Band Input Tensor Shape: {tuple(X_10band.shape)} (Reflectance scale: [0.0, 1.0])')

## Step 5 — Load Models: Baseline, Stage 1 Fine-Tuned, and Stage 2 Ultra-Res Refiner

In [ ]:
import mlstac

MODEL_DIR = '/content/model/SEN2SRLite'
os.makedirs(MODEL_DIR, exist_ok=True)
mlstac.download(
    file='https://huggingface.co/tacofoundation/SEN2SR/resolve/main/SEN2SRLite/main/mlm.json',
    output_dir=MODEL_DIR
)

# 1. Pretrained Baseline (2.5m)
model_baseline = mlstac.load(MODEL_DIR).compiled_model(device=device).eval()
print('1. Pretrained Baseline 10-Band Model (2.5m) loaded.')

# 2. Fine-Tuned Stage 1 Model (2.5m)
model_stage1 = mlstac.load(MODEL_DIR).compiled_model(device=device)
w_stage1 = torch.load(STAGE1_CKPT, map_location=device)
model_stage1.sr_model.sr_model.load_state_dict(w_stage1)
model_stage1 = model_stage1.to(device).eval()
print('2. Stage 1 Fine-Tuned 10-Band Model (2.5m) loaded.')

# 3. Stage 2 Cascaded Spatial Refiner (~1.25m)
class ResidualDenseBlock(nn.Module):
    def __init__(self, channels=48, growth_rate=24):
        super().__init__()
        self.c1 = nn.Conv2d(channels, growth_rate, 3, padding=1)
        self.c2 = nn.Conv2d(channels + growth_rate, growth_rate, 3, padding=1)
        self.c3 = nn.Conv2d(channels + 2 * growth_rate, channels, 3, padding=1)
        self.act = nn.LeakyReLU(0.05, inplace=True)

    def forward(self, x):
        x1 = self.act(self.c1(x))
        x2 = self.act(self.c2(torch.cat([x, x1], 1)))
        x3 = self.c3(torch.cat([x, x1, x2], 1))
        return x + (0.2 * x3)

class Stage2SpatialRefiner(nn.Module):
    def __init__(self, in_channels=4, out_channels=4, feat_channels=48, num_blocks=5):
        super().__init__()
        self.head = nn.Conv2d(in_channels, feat_channels, 3, padding=1)
        self.blocks = nn.ModuleList([ResidualDenseBlock(feat_channels) for _ in range(num_blocks)])
        self.body_conv = nn.Conv2d(feat_channels, feat_channels, 3, padding=1)
        self.upsampler = nn.Sequential(
            nn.Conv2d(feat_channels, out_channels * 4, 3, padding=1),
            nn.PixelShuffle(2)
        )
        self.skip_interp = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)

    def forward(self, x):
        f_in = self.head(x)
        f = f_in
        for b in self.blocks:
            f = b(f)
        f = f_in + self.body_conv(f)
        sr = self.upsampler(f)
        skip = self.skip_interp(x)
        return torch.clamp(sr + skip, 0.0, 1.0)

model_stage2 = Stage2SpatialRefiner(in_channels=4, out_channels=4, feat_channels=48, num_blocks=5).to(device)
if os.path.exists(STAGE2_CKPT):
    model_stage2.load_state_dict(torch.load(STAGE2_CKPT, map_location=device))
    print('3. Stage 2 Cascaded Spatial Refiner (~1.25m) loaded successfully!')
else:
    print('3. Stage 2 checkpoint not yet trained; using initialized model for preview.')
model_stage2 = model_stage2.eval()

## Step 6 — Multi-Scale Seamless Sliding Window Inference

In [ ]:
def run_sliding_window_stage1(model, X, patch_size=128, stride=48, scale=4):
    ps_hr = patch_size * scale
    _, H, W = X.shape
    pad_h = max(0, patch_size - H)
    pad_w = max(0, patch_size - W)
    X_pad = F.pad(X, (0, pad_w, 0, pad_h), mode='replicate')
    Hp, Wp = X_pad.shape[-2:]
    
    wy = torch.hann_window(ps_hr, periodic=False, device=device)
    wx = torch.hann_window(ps_hr, periodic=False, device=device)
    window = torch.outer(wy, wx).unsqueeze(0).clamp(min=1e-3)
    
    out = torch.zeros((10, Hp * scale, Wp * scale), device=device, dtype=X.dtype)
    weights = torch.zeros((1, Hp * scale, Wp * scale), device=device, dtype=X.dtype)
    
    y_steps = list(range(0, Hp - patch_size + 1, stride))
    if (Hp - patch_size) not in y_steps:
        y_steps.append(Hp - patch_size)
    x_steps = list(range(0, Wp - patch_size + 1, stride))
    if (Wp - patch_size) not in x_steps:
        x_steps.append(Wp - patch_size)
        
    with torch.inference_mode():
        for y0 in y_steps:
            for x0 in x_steps:
                patch = X_pad[:, y0:y0+patch_size, x0:x0+patch_size].unsqueeze(0)
                pred = model(patch).squeeze(0)
                oy, ox = y0 * scale, x0 * scale
                out[:, oy:oy+ps_hr, ox:ox+ps_hr] += pred * window
                weights[:, oy:oy+ps_hr, ox:ox+ps_hr] += window
                
    super_res = out / weights.clamp_min(1e-6)
    return super_res[:, :H*scale, :W*scale]

print('Executing 4-Way Inference across the scene...')
t0 = time.time()

# 1. Normal Input (10m) upscaled 8x to match final 1.25m grid
X_8x_input = F.interpolate(X_10band.unsqueeze(0), scale_factor=8, mode='bilinear', align_corners=False).squeeze(0)

# 2. Pretrained Baseline (2.5m) upscaled 2x to match 1.25m grid
superX_base_2dot5m = run_sliding_window_stage1(model_baseline, X_10band)
superX_base_8x = F.interpolate(superX_base_2dot5m.unsqueeze(0), scale_factor=2, mode='bilinear', align_corners=False).squeeze(0)

# 3. Fine-Tuned Stage 1 (2.5m) upscaled 2x to match 1.25m grid
superX_stage1_2dot5m = run_sliding_window_stage1(model_stage1, X_10band)
superX_stage1_8x = F.interpolate(superX_stage1_2dot5m.unsqueeze(0), scale_factor=2, mode='bilinear', align_corners=False).squeeze(0)

# 4. Stage 2 Cascaded Refiner (1.25m, ~1m GSD)
# Refiner runs on RGBN bands (B02, B03, B04, B08 -> indices [0, 1, 2, 6])
rgbn_2dot5m = superX_stage1_2dot5m[[0, 1, 2, 6]].unsqueeze(0)
with torch.inference_mode():
    rgbn_1m = model_stage2(rgbn_2dot5m).squeeze(0)

# Build full 10-band 1.25m stack (SWIR/RedEdge bilinearly refined, RGBN deep refined)
superX_1m = superX_stage1_8x.clone()
superX_1m[[0, 1, 2, 6]] = rgbn_1m

print(f'4-Way Inference completed in {(time.time() - t0):.2f} seconds!')
print(f'Final Output Shape (1.25m): {tuple(superX_1m.shape)}')

## Step 7 — 4-Way Visual Benchmark & Spatial Gradient Acuity Table

In [ ]:
def to_rgb_view(stack_10b):
    # B04 (Red: 2), B03 (Green: 1), B02 (Blue: 0)
    rgb = stack_10b[[2, 1, 0]].permute(1, 2, 0).detach().cpu().numpy()
    lo, hi = np.percentile(rgb, [2, 98])
    return np.clip((rgb - lo) / max(hi - lo, 1e-6), 0, 1)

def calc_ndvi(stack_10b):
    b08 = stack_10b[6].detach().cpu().numpy()
    b04 = stack_10b[2].detach().cpu().numpy()
    return np.clip((b08 - b04) / (b08 + b04 + 1e-6), -1, 1)

def calc_ndwi(stack_10b):
    b03 = stack_10b[1].detach().cpu().numpy()
    b08 = stack_10b[6].detach().cpu().numpy()
    return np.clip((b03 - b08) / (b03 + b08 + 1e-6), -1, 1)

rgb_in = to_rgb_view(X_8x_input)
rgb_base = to_rgb_view(superX_base_8x)
rgb_s1 = to_rgb_view(superX_stage1_8x)
rgb_1m = to_rgb_view(superX_1m)

ndvi_in = calc_ndvi(X_8x_input)
ndvi_base = calc_ndvi(superX_base_8x)
ndvi_s1 = calc_ndvi(superX_stage1_8x)
ndvi_1m = calc_ndvi(superX_1m)

# 1. 4-Way Side-by-Side Visual Plot
fig, axes = plt.subplots(2, 4, figsize=(22, 10))

models_rgb = [
    ('1. Normal Input (10m Upscaled)', rgb_in),
    ('2. Baseline SEN2SR (2.5m)', rgb_base),
    ('3. Fine-Tuned Stage 1 (2.5m, Ours)', rgb_s1),
    ('4. Cascaded Ultra-Res (~1.25m, Ours)', rgb_1m)
]

for idx, (title, img) in enumerate(models_rgb):
    axes[0, idx].imshow(img)
    axes[0, idx].set_title(title, fontsize=11, fontweight='bold')
    axes[0, idx].axis('off')

models_ndvi = [
    ('NDVI: 10m Input', ndvi_in),
    ('NDVI: 2.5m Baseline', ndvi_base),
    ('NDVI: 2.5m Stage 1', ndvi_s1),
    ('NDVI: ~1.25m Cascaded Ultra-Res', ndvi_1m)
]

for idx, (title, img) in enumerate(models_ndvi):
    im = axes[1, idx].imshow(img, cmap='RdYlGn', vmin=-1, vmax=1)
    axes[1, idx].set_title(title, fontsize=11, fontweight='bold')
    axes[1, idx].axis('off')

plt.tight_layout()
comp_path = os.path.join(OUTPUT_DIR, 'SEN2SR_4Way_1Meter_Comparison.png')
plt.savefig(comp_path, dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print(f'4-Way Comparison saved to: {comp_path}')

# 2. Quantitative 4-Way Acuity Metrics Table
def get_edge_acuity(arr):
    t = torch.from_numpy(arr).float().unsqueeze(0).unsqueeze(0)
    sx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
    sy = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3)
    mag = torch.sqrt(F.conv2d(t, sx, padding=1)**2 + F.conv2d(t, sy, padding=1)**2 + 1e-8)
    return float(mag.mean().item())

metrics_4way = [
    {
        'Resolution Tier': '1. Normal Input (10m)',
        'Ground Resolution': '10.0 meters',
        'RGB Edge Acuity': round(get_edge_acuity(rgb_in.mean(axis=-1)), 5),
        'NDVI Edge Acuity': round(get_edge_acuity(ndvi_in), 5),
        'Sharpness Gain vs 10m': '0.00%'
    },
    {
        'Resolution Tier': '2. Pretrained Baseline (2.5m)',
        'Ground Resolution': '2.5 meters',
        'RGB Edge Acuity': round(get_edge_acuity(rgb_base.mean(axis=-1)), 5),
        'NDVI Edge Acuity': round(get_edge_acuity(ndvi_base), 5),
        'Sharpness Gain vs 10m': f"{((get_edge_acuity(rgb_base.mean(axis=-1)) - get_edge_acuity(rgb_in.mean(axis=-1))) / get_edge_acuity(rgb_in.mean(axis=-1)) * 100):+.2f}%"
    },
    {
        'Resolution Tier': '3. Fine-Tuned Stage 1 (2.5m, Ours)',
        'Ground Resolution': '2.5 meters',
        'RGB Edge Acuity': round(get_edge_acuity(rgb_s1.mean(axis=-1)), 5),
        'NDVI Edge Acuity': round(get_edge_acuity(ndvi_s1), 5),
        'Sharpness Gain vs 10m': f"{((get_edge_acuity(rgb_s1.mean(axis=-1)) - get_edge_acuity(rgb_in.mean(axis=-1))) / get_edge_acuity(rgb_in.mean(axis=-1)) * 100):+.2f}%"
    },
    {
        'Resolution Tier': '4. Cascaded Ultra-Res (~1.25m, Ours)',
        'Ground Resolution': '1.25 meters',
        'RGB Edge Acuity': round(get_edge_acuity(rgb_1m.mean(axis=-1)), 5),
        'NDVI Edge Acuity': round(get_edge_acuity(ndvi_1m), 5),
        'Sharpness Gain vs 10m': f"{((get_edge_acuity(rgb_1m.mean(axis=-1)) - get_edge_acuity(rgb_in.mean(axis=-1))) / get_edge_acuity(rgb_in.mean(axis=-1)) * 100):+.2f}%"
    }
]

df_4way = pd.DataFrame(metrics_4way).set_index('Resolution Tier')
print('\n' + '='*75)
print('MAJOR PROJECT: 4-WAY MULTI-SCALE QUANTITATIVE BENCHMARK TABLE')
print('='*75)
display(df_4way)

csv_4way_path = os.path.join(OUTPUT_DIR, 'SEN2SR_4Way_1Meter_Metrics.csv')
df_4way.to_csv(csv_4way_path)
print(f'\nMetrics table saved to: {csv_4way_path}')